# `date_recorded` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `date` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'date_recorded'
feature_metadata = {'order': 2, 'name': 'date_recorded', 'audit_type': 'date', 'role': 'candidate', 'disposition': 'replace with elapsed days from a fixed competition-era reference', 'finding': 'Every supplied date parses, but collection timing is concentrated in survey waves.', 'decision': 'Replace the raw date with days_since_recorded from 2015-02-02; do not derive separate calendar components.', 'reference_date': '2015-02-02', 'reference_basis': 'Earliest surviving official DrivenData Pump It Up community post; not asserted as a verified launch timestamp.', 'reference_evidence': 'https://community.drivendata.org/t/about-the-pump-it-up-data-mining-the-water-table-category/63', 'risk': 'Recording time can proxy survey operations and geography rather than waterpoint condition.', 'related': [{'feature': 'construction_year', 'reason': 'Together they define waterpoint age at observation.'}, {'feature': 'region', 'reason': 'Survey waves may have moved through regions at different times.'}, {'feature': 'installer', 'reason': 'Installer activity and recorded construction cohorts may be time-dependent.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for date_recorded.


## Supported target evidence


In [2]:
dates = pd.to_datetime(training_data[feature], errors="coerce")
for component_name, component in (
    ("recording year", dates.dt.year),
    ("recording month", dates.dt.month),
):
    profile = pd.crosstab(
        component,
        training_data["status_group"],
        normalize="index",
    ).mul(100).round(2)
    profile.insert(0, "rows", component.value_counts())
    print(component_name)
    display(profile)


recording year


status_group,rows,functional,functional needs repair,non functional
date_recorded,,,,
2002,1,100.00,0.00,0.00
2004,30,33.33,3.33,63.33
2011,28674,56.57,6.32,37.11
2012,6424,48.66,5.64,45.70
2013,24271,53.16,8.83,38.02


recording month


status_group,rows,functional,functional needs repair,non functional
date_recorded,,,,
1,6354,41.06,12.70,46.24
2,12402,55.21,7.53,37.26
3,17936,61.65,5.02,33.33
4,3970,51.64,11.03,37.33
5,336,60.12,2.98,36.90
6,346,78.03,2.60,19.36
7,6928,50.16,7.92,41.92
8,3364,51.10,7.52,41.38
9,328,65.24,3.05,31.71


## Observation

Every supplied date parses, but collection timing is concentrated in survey waves.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Replace the raw date with days_since_recorded from 2015-02-02; do not derive separate calendar components.

**Risk to carry forward:** Recording time can proxy survey operations and geography rather than waterpoint condition.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
date_recorded,candidate,replace with elapsed days from a fixed competition-era reference,"Every supplied date parses, but collection tim...",Replace the raw date with days_since_recorded...,Recording time can proxy survey operations and...
